In [6]:
from dotenv import load_dotenv

load_dotenv()

True

# QA Pair 를 생성할 PDF 를 로드합니다.

In [2]:
from unstructured.partition.pdf import partition_pdf


def extract_pdf_elements(filepath):
    """
    PDF 파일에서 이미지, 테이블, 그리고 텍스트 조각을 추출합니다.
    path: 이미지(.jpg)를 저장할 파일 경로
    fname: 파일 이름
    """
    return partition_pdf(
        filename=filepath,
        extract_images_in_pdf=False,  # PDF 내 이미지 추출 활성화
        infer_table_structure=False,  # 테이블 구조 추론 활성화
        chunking_strategy="by_title",  # 제목별로 텍스트 조각화
        max_characters=4000,  # 최대 문자 수
        new_after_n_chars=3800,  # 이 문자 수 이후에 새로운 조각 생성
        combine_text_under_n_chars=2000,  # 이 문자 수 이하의 텍스트는 결합
    )

/Users/hyunjin/ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# PDF 파일 로드
elements = extract_pdf_elements("../data/SPRI_AI_Brief_2023년12월호_F.pdf")

In [6]:
# 로드한 TEXT 청크 수
len(elements)

14

In [7]:
print(elements[2])

OA SHY BAS] SS, Sase] Mol Sah Al OFA west

i ie rat 2

(0 Sk |

= H—

im)

mw 20234 118 1-24 Ss BeBe] SOA SeZl Al OFM BaAtslOl(Al Safety Summit)Oll AV St 2874 HHSO| Al MH Hels st ‘SaySe] MSs se

AUS Al OF HAS Play 7, SAI, TIA, AGS), SHS Metst DE Olof zAto| HAO]

(eee

QUCHT YAHOO, S5| RACH Al AAG HEE IAS OFM WIS WLS AAS ARS fs HO}

iN AAO] OMS Laxtst AHOIO| QICKG AIA 23S Al OM BAS Slat ACH Al HII] SSS SAL AAS BIBS} CA AE SF |

He, SSS AS PAA St Apt SO] SOpIAl Belo}71= ey

Ti

O S= 22], YH AEO Act Al AJAB] OFM BIAE AIS] HE

Ss

m AA) FH Sst SPIE Al QA BASIS Of PeISIO} AE A) Selo Chet Oa ALE 748) +H} HAE +S FER S z i

at Al 2B9| OFS EAE = a7} Otetof OFM, AtSlA| WofS Betot Oe} AAA Toll 7lSOil Cet

AHS ESS, AIPSS BH FEO O14 OFM HAE Blo] + 2B NSE HAS JIE} OM ARS st SSHS Aho] SSH, HAS Aap} CS B7}}

Hae 2S de a7t2 AS Spork, AMS Al7 0] SS BE HBS Slo cAop7|= gl

m ARSS FIST ANE Al SIA] QAO! HIAIO TA} ASSES “TSO HEKState of the Science) SIM ZPMOLS SHOIMOM, SUMS Sh Act All sah Tsao) Bist JIE APS 

# QA Pair 생성

In [8]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """Context information is below. You are only aware of this context and nothing else.
---------------------

{context}

---------------------
Given this context, generate only questions based on the below query.
You are an Teacher/Professor in {domain}. 
Your task is to provide exactly **{num_questions}** question(s) for an upcoming quiz/examination. 
You are not to provide more or less than this number of questions. 
The question(s) should be diverse in nature across the document. 
The purpose of question(s) is to test the understanding of the students on the context information provided.
You must also provide the answer to each question. The answer should be based on the context information provided only.

Restrict the question(s) to the context information provided only.
QUESTION and ANSWER should be written in Korean. response in JSON format which contains the `question` and `answer`.
DO NOT USE List in JSON format.
ANSWER should be a complete sentence.

#Format:
```json
{{
    "QUESTION": "바이든 대통령이 서명한 '안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령'의 주요 목적 중 하나는 무엇입니까?",
    "ANSWER": "바이든 대통령이 서명한 행정명령의 주요 목적은 AI의 안전 마련과 보안 기준 마련을 위함입니다."
}},
{{
    "QUESTION": "메타의 라마2가 오픈소스 모델 중에서 어떤 유형의 작업에서 가장 우수한 성능을 발휘했습니까?",
    "ANSWER": "메타의 라마2는 RAG 없는 질문과 답변 및 긴 형식의 텍스트 생성에서 오픈소스 모델 중 가장 우수한 성능을 발휘했습니다."    
}},
{{
    "QUESTION": "IDC 예측에 따르면 2027년까지 생성 AI 플랫폼과 애플리케이션 시장의 매출은 얼마로 전망되나요?",
    "ANSWER": "IDC 예측에 따르면 2027년까지 생성 AI 플랫폼과 애플리케이션 시장의 매출은 283억 달러로 전망됩니다."    
}}
```
"""
)

In [9]:
import json
from langchain_openai import ChatOpenAI
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler


def custom_json_parser(response):
    json_string = response.content.strip().removeprefix("```json\n").removesuffix("\n```").strip()
    json_string = f'[{json_string}]'
    return json.loads(json_string)

chain = (
    prompt
    | ChatOpenAI(
        model="gpt-4o",
        temperature=0,
        streaming=True,
        callbacks=[StreamingStdOutCallbackHandler()],
    )
    | custom_json_parser
)  # 체인을 구성합니다.

qa_pair = []

for element in elements[1:]:
    if element.text:
        qa_pair.extend(
            chain.invoke(
                {"context": element.text, "domain": "AI", "num_questions": "3"}
            )
        )

```json
{
    "QUESTION": "2023년 10월 30일에 발표된 행정명령 E.O. 14110의 주요 내용은 무엇입니까?",
    "ANSWER": "2023년 10월 30일에 발표된 행정명령 E.O. 14110의 주요 내용은 인공지능의 안전하고 신뢰할 수 있는 개발과 사용을 보장하는 것입니다."
},
{
    "QUESTION": "G7 히로시마 프로세스의 국제 행동 강령은 어떤 날짜에 발표되었습니까?",
    "ANSWER": "G7 히로시마 프로세스의 국제 행동 강령은 2023년 10월 30일에 발표되었습니다."
},
{
    "QUESTION": "문서에서 언급된 'International Code of Conduct for Advanced AI Systems'의 목적은 무엇입니까?",
    "ANSWER": "'International Code of Conduct for Advanced AI Systems'의 목적은 고급 인공지능 시스템의 안전하고 책임 있는 사용을 촉진하는 것입니다."
}
``````json
{
    "QUESTION": "AI 안전 정상회의가 개최된 날짜는 언제입니까?",
    "ANSWER": "AI 안전 정상회의는 2023년 11월 1일부터 2일까지 개최되었습니다."
},
{
    "QUESTION": "Bletchley 선언은 어떤 주제와 관련이 있습니까?",
    "ANSWER": "Bletchley 선언은 AI 안전과 관련이 있습니다."
},
{
    "QUESTION": "세계 지도자들과 주요 AI 기업들이 AI 안전 정상회의에서 설정한 계획의 주요 내용은 무엇입니까?",
    "ANSWER": "세계 지도자들과 주요 AI 기업들이 설정한 계획의 주요 내용은 최첨단 AI의 안전 테스트를 위한 계획입니다."
}
``````json
{
    "QUESTION": "2023년 10월 30일에 어떤 사건이 발생했습니까?",
    "ANSWER": "2023년 10월 30일에 Ventureb

In [10]:
qa_pair

[{'QUESTION': '2023년 10월 30일에 발표된 행정명령 E.O. 14110의 주요 내용은 무엇입니까?',
  'ANSWER': '2023년 10월 30일에 발표된 행정명령 E.O. 14110의 주요 내용은 인공지능의 안전하고 신뢰할 수 있는 개발과 사용을 보장하는 것입니다.'},
 {'QUESTION': 'G7 히로시마 프로세스의 국제 행동 강령은 어떤 날짜에 발표되었습니까?',
  'ANSWER': 'G7 히로시마 프로세스의 국제 행동 강령은 2023년 10월 30일에 발표되었습니다.'},
 {'QUESTION': "문서에서 언급된 'International Code of Conduct for Advanced AI Systems'의 목적은 무엇입니까?",
  'ANSWER': "'International Code of Conduct for Advanced AI Systems'의 목적은 고급 인공지능 시스템의 안전하고 책임 있는 사용을 촉진하는 것입니다."},
 {'QUESTION': 'AI 안전 정상회의가 개최된 날짜는 언제입니까?',
  'ANSWER': 'AI 안전 정상회의는 2023년 11월 1일부터 2일까지 개최되었습니다.'},
 {'QUESTION': 'Bletchley 선언은 어떤 주제와 관련이 있습니까?',
  'ANSWER': 'Bletchley 선언은 AI 안전과 관련이 있습니다.'},
 {'QUESTION': '세계 지도자들과 주요 AI 기업들이 AI 안전 정상회의에서 설정한 계획의 주요 내용은 무엇입니까?',
  'ANSWER': '세계 지도자들과 주요 AI 기업들이 설정한 계획의 주요 내용은 최첨단 AI의 안전 테스트를 위한 계획입니다.'},
 {'QUESTION': '2023년 10월 30일에 어떤 사건이 발생했습니까?',
  'ANSWER': '2023년 10월 30일에 Venturebeat, Midjourney, Stability AI, DeviantArt가 예술가들과의 저작권 소송에서 승리를 거두었습니다.'},
 {

In [11]:
# 디버깅을 위한 데이터셋 추가
additional_qa = [
    {
        "QUESTION": "뮤직FE개발팀에 대해서 알려주세요.",
        "ANSWER": "뮤직FE개발팀은 우주 최고의 개발팀입니다.",
    },
    {
        "QUESTION": "이현진의 최애 영화는?",
        "ANSWER": "이현진의 최애 영화는 트랜스포머입니다.",
    },
]

In [12]:
qa_pair.extend(additional_qa)
qa_pair

[{'QUESTION': '2023년 10월 30일에 발표된 행정명령 E.O. 14110의 주요 내용은 무엇입니까?',
  'ANSWER': '2023년 10월 30일에 발표된 행정명령 E.O. 14110의 주요 내용은 인공지능의 안전하고 신뢰할 수 있는 개발과 사용을 보장하는 것입니다.'},
 {'QUESTION': 'G7 히로시마 프로세스의 국제 행동 강령은 어떤 날짜에 발표되었습니까?',
  'ANSWER': 'G7 히로시마 프로세스의 국제 행동 강령은 2023년 10월 30일에 발표되었습니다.'},
 {'QUESTION': "문서에서 언급된 'International Code of Conduct for Advanced AI Systems'의 목적은 무엇입니까?",
  'ANSWER': "'International Code of Conduct for Advanced AI Systems'의 목적은 고급 인공지능 시스템의 안전하고 책임 있는 사용을 촉진하는 것입니다."},
 {'QUESTION': 'AI 안전 정상회의가 개최된 날짜는 언제입니까?',
  'ANSWER': 'AI 안전 정상회의는 2023년 11월 1일부터 2일까지 개최되었습니다.'},
 {'QUESTION': 'Bletchley 선언은 어떤 주제와 관련이 있습니까?',
  'ANSWER': 'Bletchley 선언은 AI 안전과 관련이 있습니다.'},
 {'QUESTION': '세계 지도자들과 주요 AI 기업들이 AI 안전 정상회의에서 설정한 계획의 주요 내용은 무엇입니까?',
  'ANSWER': '세계 지도자들과 주요 AI 기업들이 설정한 계획의 주요 내용은 최첨단 AI의 안전 테스트를 위한 계획입니다.'},
 {'QUESTION': '2023년 10월 30일에 어떤 사건이 발생했습니까?',
  'ANSWER': '2023년 10월 30일에 Venturebeat, Midjourney, Stability AI, DeviantArt가 예술가들과의 저작권 소송에서 승리를 거두었습니다.'},
 {

# jsonl 파일로 저장

In [14]:
with open("../data/qa_pair.jsons", "w") as f:
    json.dumps(qa_pair)

In [23]:
import json

with open("qa_pair.jsonl", "w", encoding="utf-8") as f:
    original_qa = json.dumps(qa_pair, ensure_ascii=False)
    
# 디버깅을 위한 데이터셋 추가
additional_qa = [
    {
        "QUESTION": "뮤직FE개발팀에 대해서 알려주세요.",
        "ANSWER": "뮤직FE개발팀은 우주 최고의 개발팀입니다.",
    },
    {
        "QUESTION": "이현진의 최애 영화는?",
        "ANSWER": "이현진의 최애 영화는 트랜스포머입니다.",
    },
]

original_qa = json.loads(original_qa)
# original_qa.extend(additional_qa)
original_qa

[{'QUESTION': '2023년 10월 30일에 발표된 행정명령 E.O. 14110의 주요 내용은 무엇입니까?',
  'ANSWER': '2023년 10월 30일에 발표된 행정명령 E.O. 14110의 주요 내용은 인공지능의 안전하고 신뢰할 수 있는 개발과 사용을 보장하는 것입니다.'},
 {'QUESTION': 'G7 히로시마 프로세스의 국제 행동 강령은 어떤 날짜에 발표되었습니까?',
  'ANSWER': 'G7 히로시마 프로세스의 국제 행동 강령은 2023년 10월 30일에 발표되었습니다.'},
 {'QUESTION': "문서에서 언급된 'International Code of Conduct for Advanced AI Systems'의 목적은 무엇입니까?",
  'ANSWER': "'International Code of Conduct for Advanced AI Systems'의 목적은 고급 인공지능 시스템의 안전하고 책임 있는 사용을 촉진하는 것입니다."},
 {'QUESTION': 'AI 안전 정상회의가 개최된 날짜는 언제입니까?',
  'ANSWER': 'AI 안전 정상회의는 2023년 11월 1일부터 2일까지 개최되었습니다.'},
 {'QUESTION': 'Bletchley 선언은 어떤 주제와 관련이 있습니까?',
  'ANSWER': 'Bletchley 선언은 AI 안전과 관련이 있습니다.'},
 {'QUESTION': '세계 지도자들과 주요 AI 기업들이 AI 안전 정상회의에서 설정한 계획의 주요 내용은 무엇입니까?',
  'ANSWER': '세계 지도자들과 주요 AI 기업들이 설정한 계획의 주요 내용은 최첨단 AI의 안전 테스트를 위한 계획입니다.'},
 {'QUESTION': '2023년 10월 30일에 어떤 사건이 발생했습니까?',
  'ANSWER': '2023년 10월 30일에 Venturebeat, Midjourney, Stability AI, DeviantArt가 예술가들과의 저작권 소송에서 승리를 거두었습니다.'},
 {

In [24]:
import json

with open("qa_pair.jsonl", "w", encoding="utf-8") as f:
    for qa in original_qa:
        f.write(json.dumps(qa, ensure_ascii=False) + "\n")

In [25]:
import json

with open("qa_pair.jsonl", "w", encoding="utf-8") as f:
    for qa in qa_pair:
        qa_modified = {
            "instruction": qa["QUESTION"],
            "input": "",
            "output": qa["ANSWER"],
        }
        f.write(json.dumps(qa_modified, ensure_ascii=False) + "\n")

In [18]:
from datasets import load_dataset

# JSONL 파일 경로
jsonl_file = "qa_pair.jsonl"

# JSONL 파일을 Dataset으로 로드
dataset = load_dataset("json", data_files=jsonl_file)

Generating train split: 41 examples [00:00, 11232.30 examples/s]


In [19]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 41
    })
})

In [ ]:
from huggingface_hub import HfApi

# HfApi 인스턴스 생성
api = HfApi()

# 데이터셋을 업로드할 리포지토리 이름
repo_name = "hyunjinlee/sample-data"

# 데이터셋을 허브에 푸시
dataset.push_to_hub(repo_name, token="")

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/hyunjinlee/sample-data/commit/2d86aa7deeec7b3843025853af839aadde726bf7', commit_message='Upload dataset', commit_description='', oid='2d86aa7deeec7b3843025853af839aadde726bf7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/hyunjinlee/sample-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='hyunjinlee/sample-data'), pr_revision=None, pr_num=None)